In [5]:
!pip install grad-cam


     ---------------------------------------- 0.0/7.8 MB ? eta -:--:--
     -- ------------------------------------- 0.5/7.8 MB 10.1 MB/s eta 0:00:01
     ---------- ----------------------------- 2.1/7.8 MB 7.2 MB/s eta 0:00:01
     ----------------- ---------------------- 3.4/7.8 MB 7.0 MB/s eta 0:00:01
     ------------------------ --------------- 4.7/7.8 MB 6.9 MB/s eta 0:00:01
     -------------------------------- ------- 6.3/7.8 MB 6.7 MB/s eta 0:00:01
     ---------------------------------------- 7.8/7.8 MB 6.7 MB/s  0:00:01
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for grad-cam: filename=grad_cam-1.5.5-py3-none-any.whl size=44339 sha256=e23486035fcbe6c421dcd80ac026c660645944a1e6f6

In [18]:
import torch
import numpy as np
import cv2
from PIL import Image
from torchvision import transforms, models
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [19]:
model = models.mobilenet_v2(pretrained=False)
model.classifier = torch.nn.Sequential(
    torch.nn.Linear(model.last_channel, 128),
    torch.nn.ReLU(),
    torch.nn.Dropout(0.3),
    torch.nn.Linear(128, 1),
    torch.nn.Sigmoid()
)
model.load_state_dict(torch.load("mobilenetv2_pneumonia.pth", map_location=device))
model.to(device)
model.eval()



c:\Users\Admin\anaconda3\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Admin\anaconda3\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [20]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])



In [23]:
img_path = "data_split/test/Normal/Normal-9.png"  # your X-ray
img = Image.open(img_path).convert("RGB")
input_tensor = transform(img).unsqueeze(0).to(device)


In [25]:
target_layers = [model.features[-1]]  

cam = GradCAM(model=model, target_layers=target_layers)
grayscale_cam = cam(input_tensor=input_tensor)[0]


In [29]:
img_resized = img.resize((224,224))
img_np = np.array(img_resized)/255.0

cam_image = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)


In [30]:
cv2.imwrite("grad.png", cam_image)
print("Grad-CAM saved as gradcam_output.png")

Grad-CAM saved as gradcam_output.png
